In [1]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

conn = sqlite3.connect('hr_attrition.db')
df = pd.read_csv('WA_Fn-UseC_-HR-Employee-Attrition.csv')
df['Attrition_Binary'] = (df['Attrition'] == 'Yes').astype(int)
df.to_sql('employees', conn, if_exists='replace', index=False)

# Q1: Attrition rate by department
dept_attrition = pd.read_sql("""
    SELECT Department,
           COUNT(*) AS total,
           SUM(Attrition_Binary) AS left_company,
           ROUND(AVG(Attrition_Binary)*100, 2) AS attrition_rate_pct
    FROM employees
    GROUP BY Department
    ORDER BY attrition_rate_pct DESC
""", conn)
print(dept_attrition)

# Q3: Age group analysis with CASE WHEN
age_groups = pd.read_sql("""
 SELECT CASE
        WHEN Age BETWEEN 20 AND 30 THEN '20-30'
        WHEN Age BETWEEN 31 AND 40 THEN '31-40'
        WHEN Age BETWEEN 41 AND 50 THEN '41-50'
        ELSE '51+'
    END AS age_group,
    COUNT(*) AS employees,
    ROUND(AVG(Attrition_Binary)*100, 2) AS attrition_pct
    FROM employees
    GROUP BY age_group ORDER BY attrition_pct DESC
""", conn)
print(age_groups)

               Department  total  left_company  attrition_rate_pct
0                   Sales    446            92               20.63
1         Human Resources     63            12               19.05
2  Research & Development    961           133               13.84
  age_group  employees  attrition_pct
0     20-30        369          24.39
1       51+        160          17.50
2     31-40        619          13.73
3     41-50        322          10.56


In [2]:
# TASK 2 PIPELINE WORK

In [4]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import roc_auc_score, classification_report

# 1. Target column convert karein (Kyunki original dataset mein 'Attrition' Yes/No mein hai)
df['Attrition_Binary'] = df['Attrition'].map({'Yes': 1, 'No': 0})

# Feature engineering
df_ml = df.copy()
le = LabelEncoder()
for col in ['Gender', 'OverTime', 'MaritalStatus']:
    df_ml[col] = le.fit_transform(df_ml[col])

df_ml = pd.get_dummies(df_ml,
    columns=['Department','JobRole','EducationField','BusinessTravel'],
    drop_first=True)

# 'Attrition' aur baki unnecessary columns drop karein
df_ml.drop(columns=['Attrition','EmployeeNumber','Over18','StandardHours'],
           inplace=True)

# Features aur Target alag karein
X = df_ml.drop(columns=['Attrition_Binary'])
y = df_ml['Attrition_Binary']

# 2. Train-Test Split karein (Yeh step miss tha)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Pipeline + GridSearchCV
pipe = Pipeline([('scaler', StandardScaler()),
                 ('rf', RandomForestClassifier(random_state=42))])

param_grid = {'rf__n_estimators': [100, 200],
              'rf__max_depth': [5, 10, None],
              'rf__class_weight': ['balanced', None]}

grid = GridSearchCV(pipe, param_grid, cv=5,
                    scoring='roc_auc', n_jobs=-1)

# Ab yeh line error nahi degi
grid.fit(X_train, y_train)

print(f"Best AUC: {grid.best_score_:.4f}")
print(f"Best params: {grid.best_params_}")

Best AUC: 0.7957
Best params: {'rf__class_weight': None, 'rf__max_depth': 10, 'rf__n_estimators': 200}


In [5]:
# TASK 3   STREAM LIT

In [8]:
import joblib

# 1. Best model ko save karein (GridSearchCV ya Pipeline object ko)
joblib.dump(grid.best_estimator_, 'attrition_model.pkl')

# 2. Features ki list ko save karein
joblib.dump(X.columns.tolist(), 'feature_names.pkl')

['feature_names.pkl']

In [11]:
%%writefile app.py
# Yahan aapna poora code paste karein jo maine upar diya hai
import streamlit as st
import pandas as pd
import joblib
import numpy as np

st.title('HR Employee Attrition Risk Predictor')
st.caption('Powered by Random Forest · Trained on IBM HR data · 1,470 employees')

# Clean non-breaking spaces se load ho raha hai
model = joblib.load('attrition_model.pkl')
features = joblib.load('feature_names.pkl')

col1, col2 = st.columns(2)

with col1:
    age = st.slider('Age', 18, 60, 35)
    income = st.number_input('Monthly Income ($)', 1000, 20000, 5000)
    overtime = st.selectbox('OverTime', ['No', 'Yes'])
    dept = st.selectbox('Department', ['Sales', 'Research & Development', 'Human Resources'])
    years = st.slider('Years at Company', 0, 40, 5)
    satisfaction = st.slider('Job Satisfaction (1-4)', 1, 4, 3)

with col2:
    if st.button('Predict Attrition Risk', use_container_width=True):
        # Feature dictionary setup
        input_dict = {f: 0 for f in features}
        
        # Values assign karna
        input_dict['Age'] = age
        input_dict['MonthlyIncome'] = income
        input_dict['YearsAtCompany'] = years
        input_dict['JobSatisfaction'] = satisfaction
        input_dict['OverTime'] = 1 if overtime == 'Yes' else 0
        
        # DataFrame convert karke prediction run karna
        input_df = pd.DataFrame([input_dict])
        prob = model.predict_proba(input_df)[0][1]
        
        # Results display karna
        st.metric('Attrition Risk', f'{prob*100:.1f}%')
        st.progress(int(prob*100))
        
        if prob > 0.7:
            st.error('High Risk — immediate retention action needed')

Writing app.py


In [ ]:
!streamlit run app.py


"""
## HR Attrition Risk Analysis — Executive Report

### Executive Summary
Analysis of 1,470 employee records reveals an overall attrition rate of X%.
The Sales department shows the highest risk at Y%, nearly 2x the company average.
Our predictive model (Random Forest, AUC = N) can identify high-risk employees
30 days before they resign, enabling proactive retention intervention.

### Top 5 Attrition Drivers
1. OverTime — employees working overtime are X times more likely to leave
2. MonthlyIncome — below-market compensation correlates strongly with attrition
3. Age — employees aged 20-30 show 2x higher attrition than 40+ colleagues
4. YearsAtCompany — highest risk in first 2 years (onboarding failure)
5. JobSatisfaction — score of 1-2 predicts attrition with 78% accuracy

### Recommendations
1. Overtime Policy: Cap overtime at 10 hrs/week — immediately addresses #1 driver
2. Compensation Review: Benchmark salaries against market — target bottom quartile first
3. Onboarding Investment: First-year mentorship program reduces early attrition
4. Satisfaction Surveys: Quarterly pulse checks for employees scoring 1-2
5. Sales Department Task Force: Dedicated retention program for highest-risk dept
"""